# 🛡️ LLaVA-1.5-7b Multimodal Cross-Modal Unlearning (Google Colab)
### Adaptive Relearning-Resistant Multimodal Unlearning via NPO+SAM

This notebook trains and evaluates cross-modal unlearning on the **Real LLaVA-1.5-7b** vision-language model using 4-bit QLoRA and streaming real **Flickr30k** images. 

| Requirements | Detail |
| --- | --- |
| **GPU Runtime** | Recommended: **T4 GPU** (with High-RAM) or **A100 / L4** |
| **Weights Cache** | ~15 GB downloading space (automatically managed by Colab cloud disk) |
| **Estimated Time** | ~15-20 minutes total |

In [ ]:
# ── 1. Check GPU Status ──
import subprocess, torch
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'NVIDIA' in gpu_info.stdout:
    print('✅ GPU detected successfully:')
    print('\n'.join(gpu_info.stdout.split('\n')[:12]))
else:
    print('⚠️ WARNING: No GPU detected! Go to: Runtime -> Change runtime type -> Select T4 GPU or A100.')

print(f'\nPyTorch version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'Target device   : {gpu.name} ({gpu.total_memory / 1e9:.1f} GB VRAM)')

In [ ]:
# ── 2. Clone ARMOR Repository ──
import os
REPO_URL = 'https://github.com/Angrajkarn/-ARMOR-Adaptive-Relearning-resistant-Multimodal-Unlearning.git'
REPO_DIR = '/content/ARMOR'

if not os.path.exists(REPO_DIR):
    print(f'Cloning repository from {REPO_URL}...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository folder already exists, pulling updates...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'\n✅ Working Directory changed to: {os.getcwd()}')

In [ ]:
# ── 3. Install Dependencies ──
# Ensure compatibility with LLaVA model loaders & QLoRA quantization.
print('Installing dependencies (this may take 2-3 minutes)...')
!pip install -q transformers>=4.37.0 peft>=0.6.0 datasets>=2.18.0 accelerate>=0.28.0 bitsandbytes>=0.41.0 trl>=0.8.0 rouge-score scipy scikit-learn pandas matplotlib Pillow>=10.0.0 torchvision>=0.16.0
print('✅ Dependencies installed successfully!')

In [ ]:
# ── 4. HuggingFace Authentication ──
# Gaining access to gated weights under your HuggingFace account.
from huggingface_hub import login
import os

HF_TOKEN = ""
if not HF_TOKEN:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
        print('✅ Found HF_TOKEN in Google Colab Secrets (userdata)')
    except Exception:
        HF_TOKEN = input("Please paste your HuggingFace Access Token: ").strip()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in successfully!")
else:
    print("⚠️ Proceeding without token. (Ensure models used are public and not gated)")

---  
## 🚀 Section 2: Execute Real LLaVA Multimodal Unlearning
Here we execute the compiled `run_llava_unlearn.py` command using the real model parameters.

- `--real-llava`: Streams real image-caption datasets via Flickr30k alongside the LLaVA vision encoder.
- `--qlora`: Activates model weight quantization to fit inside the standard GPU VRAM window.
- `--run-mia`: Conducts Membership Inference Audits to verify data forgetting quality post-unlearning.

In [ ]:
# ── 5. Run Multimodal Pipeline ──
# Parameters:
# --sam-rho (radius parameter for the geometrical SAM optimization: default 0.05)
# --beta-npo (NPO loss weight default: 0.1)

token_arg = f"--hf-token {HF_TOKEN}" if HF_TOKEN else ""

!python scripts/run_llava_unlearn.py \
    --real-llava \
    --qlora \
    --sam-rho 0.05 \
    --beta-npo 0.1 \
    --run-mia \
    {token_arg} \
    --output-dir "/content/drive/MyDrive/outputs/llava_npo_sam"